# GemCol Evaluation: Phase 2 (Optimized BM25 Baseline)
This uses `bm25s`, an extremely memory-efficient and fast BM25 implementation in Python that prevents the 64GB RAM crash caused by `rank_bm25`.

In [ ]:
!pip install bm25s

In [ ]:
import sys
import os
import bm25s
from tqdm import tqdm

sys.path.insert(0, '/workspace/gemcol_evaluation')

from utils import (
    load_msmarco_dev, save_run_json, evaluate_run, print_metrics, 
    save_checkpoint, load_or_compute, LatencyProfiler, ExperimentTracker, 
    sanity_check_ndcg
)

print("Loading MS MARCO dataset via HF Datasets...")
queries, qrels, corpus = load_msmarco_dev()
print(f"Loaded {len(queries)} queries and {len(corpus)} passages.")

In [ ]:
def build_bm25_index():
    doc_ids = list(corpus.keys())
    passages = list(corpus.values())
    
    print("Tokenizing corpus (this is highly optimized in bm25s)...")
    # bm25s uses a highly memory-efficient tokenizer
    corpus_tokens = bm25s.tokenize(passages)
    
    print("Building BM25 sparse index...")
    retriever = bm25s.BM25(k1=0.9, b=0.4)
    retriever.index(corpus_tokens)
    
    return retriever, doc_ids

print("Building or loading BM25 index...")
bm25_retriever, doc_ids = load_or_compute("bm25_index_and_docids", build_bm25_index)
print("✅ BM25 index ready.")

In [ ]:
prof = LatencyProfiler()
bm25_run = {}

# bm25s does batched retrieval which is lightning fast
query_ids = list(queries.keys())
query_texts = list(queries.values())

print("Tokenizing queries...")
query_tokens = bm25s.tokenize(query_texts)

with prof.time("bm25_batch"):
    print("Retrieving top 100 docs per query...")
    # bm25s returns the doc indices and their scores
    results, scores = bm25_retriever.retrieve(query_tokens, k=100)

print("Formatting results...")
for i, qid in enumerate(query_ids):
    # results[i] contains the index mapping back to doc_ids
    bm25_run[qid] = [doc_ids[idx] for idx in results[i]]

# Save the run for RRF fusion later
save_run_json(bm25_run, "/workspace/results/bm25_run.json")
save_checkpoint(bm25_run, "bm25_run_msmarco")
print("✅ Retrieval complete and saved.")

In [ ]:
metrics = evaluate_run(bm25_run, qrels)
print_metrics(metrics, system_name="BM25 Baseline (bm25s)")

# Check if the score is in the expected 0.28 - 0.30 range
sanity_check_ndcg("bm25", metrics["ndcg@10"])

print("\nLatency Profiling:")
prof.print_summary()

tracker = ExperimentTracker("/workspace/results/experiments.json")
tracker.log("BM25 baseline", config={"k1": 0.9, "b": 0.4, "backend": "bm25s"}, metrics={
    "ndcg@10": metrics["ndcg@10"],
    "mrr@10": metrics["mrr@10"],
    "recall@100": metrics["recall@100"],
    "latency_ms_total": prof.summary()["bm25_batch"]["mean_ms"]
})
print("✅ Metrics logged to experiments.json")